In [38]:
import pandas as pd
import numpy as np

# Load Sentiment Scores

In [32]:
sentiment_scores = pd.read_csv("/Users/ohungchan/Downloads/merged_sentiment_scores_2.csv")
tracks_with_lyrics = pd.read_parquet("phase2_data_cleaning/cleaned dataset/track_cleaned.parquet", columns=['track_idx', 'lyrics'])

In [33]:
# List of invalid values (non-newline ones)
invalid_values = ["", "None", "missing", "?", "-", "|Instrumentals|", "{music]", "{Intrumental}", "{Instrumental track}", "none"]

# Replace invalid values with NaN in the 'lyrics' column
tracks_with_lyrics['lyrics'] = tracks_with_lyrics['lyrics'].replace(invalid_values, np.nan)

# Use regex to replace strings that only consist of newlines (\n, \n\n, \n\n\n, etc.)
tracks_with_lyrics['lyrics'] = tracks_with_lyrics['lyrics'].replace(r'^\n+$', np.nan, regex=True)

# Drop rows where 'lyrics' column is NaN
tracks_with_lyrics.dropna(subset=['lyrics'], inplace=True)

In [34]:
combined = pd.merge(tracks_with_lyrics, sentiment_scores, on='track_idx', how='inner')

In [35]:
combined = combined.drop(columns=['lyrics'])

In [36]:
# combined['row_sum'] = combined[["joy", "calm", "sadness", "fear", "energizing", "dreamy"]].sum(axis=1)


In [37]:
# Define emotion columns
emotion_columns = ["joy", "calm", "sadness", "fear", "energizing", "dreamy"]

# Compute row-wise sum
row_sums = combined[emotion_columns].sum(axis=1)

# Normalize each row by dividing by the row sum
combined[emotion_columns] = combined[emotion_columns].div(row_sums, axis=0).fillna(0)

combined.to_csv("phase3_feature_engineering/scores/final_sentiment_scores.csv", index=False)

In [41]:
genre_scores = pd.read_csv("phase3_feature_engineering/scores/final_sentiment_scores.csv")

In [42]:
genre_columns = [
    "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack",
    "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)",
    "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"
]

genre_scores['sum'] = genre_scores[genre_columns].sum(axis=1)